In [1]:
from langchain_community.document_loaders import TextLoader
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from langchain_community.embeddings import HuggingFaceEmbeddings

c:\neurobot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2", 
    model_kwargs={"device": "cpu"})

C:\Users\daksh\AppData\Local\Temp\ipykernel_20176\3206400229.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2",
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 328.11it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
loader=TextLoader(r"C:\neurobot\Data\Diabetes\Diabetes.txt")
docs=loader.load()
text=docs[0].page_content

In [4]:
import re
from langchain_core.documents import Document   # ✅ new

pattern = r"\n(?=[A-Z][a-zA-Z ]+\n)"

sections = re.split(pattern,text)

docs = [
    Document(page_content=s.strip(), metadata={"source": "diabetes"})
    for s in sections
    if len(s.strip()) > 100
]


In [5]:
for i,doc in enumerate(docs):
    print(f"chunk {i}: \n {doc}")



chunk 0: 
 page_content='Overview
Diabetes mellitus refers to a group of diseases that affect how the body uses blood sugar (glucose). Glucose is an important source of energy for the cells that make up the muscles and tissues. It's also the brain's main source of fuel.

The main cause of diabetes varies by type. But no matter what type of diabetes you have, it can lead to excess sugar in the blood. Too much sugar in the blood can lead to serious health problems.

Chronic diabetes conditions include type 1 diabetes and type 2 diabetes. Potentially reversible diabetes conditions include prediabetes and gestational diabetes. Prediabetes happens when blood sugar levels are higher than normal. But the blood sugar levels aren't high enough to be called diabetes. And prediabetes can lead to diabetes unless steps are taken to prevent it. Gestational diabetes happens during pregnancy. But it may go away after the baby is born.' metadata={'source': 'diabetes'}
chunk 1: 
 page_content='Symptoms


In [6]:
from langchain_community.vectorstores import FAISS
from langchain_community.retrievers import BM25Retriever
from custom.hybrid import HybridRetriever
 





In [7]:
                                
dense_vectordb=FAISS.from_documents(docs,embedding)

dense_retriever=dense_vectordb.as_retriever()



In [8]:
sparse_retriever=BM25Retriever.from_documents(docs)
sparse_retriever.k=3

In [9]:
hybrid_retriever = HybridRetriever(
    dense=dense_retriever,
    sparse=sparse_retriever,
    weights=(0.7, 0.3)   # tune kar sakta
)


In [10]:
query="Who is at higher risk of developing diabetes?"
result=hybrid_retriever.get_relevant_documents(query)
for i,doc in enumerate(result):
    print(f"Rank #{i}: \n {doc}")

Rank #0: 
 page_content='Risk factors
Risk factors for diabetes depend on the type of diabetes. Family history may play a part in all types. Environmental factors and geography can add to the risk of type 1 diabetes.

Sometimes family members of people with type 1 diabetes are tested for the presence of diabetes immune system cells (autoantibodies). If you have these autoantibodies, you have an increased risk of developing type 1 diabetes. But not everyone who has these autoantibodies develops diabetes.

Race or ethnicity also may raise your risk of developing type 2 diabetes. Although it's unclear why, certain people â€” including Black, Hispanic, American Indian and Asian American people â€” are at higher risk.

Prediabetes, type 2 diabetes and gestational diabetes are more common in people who are overweight or obese.' metadata={'source': 'diabetes'}
Rank #1: 
 page_content='Overview
Diabetes mellitus refers to a group of diseases that affect how the body uses blood sugar (glucose).

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_groq.chat_models import ChatGroq
from dotenv import load_dotenv
from custom.hybrid import HybridRetrieverWrapper

wrapper = HybridRetrieverWrapper(hybrid=hybrid_retriever)

llm=ChatGroq(model="llama3-8b-8192")

prompt_template = """
You are a safe medical assistant.

Rules:
- Use ONLY the provided context
- Do NOT hallucinate
- If symptoms look serious, advise doctor visit
- Give short, clear answers

Context:
{context}

Question:
{question}

Answer:
"""

PROMPT = PromptTemplate(
    template=prompt_template,
    input_variables=["context", "question"]
)

qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=wrapper,
    chain_type="stuff",
    chain_type_kwargs={"prompt": PROMPT}
)

print(qa.invoke("Nowdays,I am urinating a lot compared to before and I lost 20 lbs of weight in last 2 months with a normal diet and feeling tired.Can you suggest me something"))


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}